In [8]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA, LLMChain
from langchain.text_splitter import CharacterTextSplitter
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.schema import Document
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

load_dotenv()
google_api_key=os.getenv('GOOGLE_API_KEY')

llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0,convert_system_message_to_human=True)

with open('sample.txt','r',encoding='utf-8') as f:
    raw_text=f.read()

chunks=CharacterTextSplitter(chunk_size=300,chunk_overlap=50).split_text(raw_text)
documents = [Document(page_content=chunk) for chunk in chunks]
embedding=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2-preview')

vectorstore=FAISS.from_documents(documents,embedding)

compressor=LLMChainExtractor.from_llm(llm)
contextcompressretriever=ContextualCompressionRetriever(base_compressor=compressor,base_retriever=vectorstore.as_retriever())

memory=ConversationBufferMemory(memory_key='chat_history',return_messages=True)

In [3]:
response1=contextcompressretriever.get_relevant_documents("Who created LangChain?")
print('\nLLM answer: ',response1)

C:\Users\DELL\AppData\Local\Temp\ipykernel_9128\1684996308.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response1=contextcompressretriever.get_relevant_documents("Who created LangChain?")



LLM answer:  [Document(metadata={}, page_content='LangChain was created by Harrison Chase.')]


In [4]:
qa_chain=RetrievalQA.from_chain_type(llm=llm,retriever=contextcompressretriever,memory=memory)

response2=qa_chain.invoke({'query':'Who created Langchain'})['result']
print('\nLLM answer: ',response2)


LLM answer:  LangChain was created by Harrison Chase.


In [5]:
qa_tool=Tool(
    name='Simple QA',
    func=qa_chain.invoke,
    description='A basic LLM that answers users queries clearly'
)

agent=initialize_agent(
    tools=[qa_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_error=True
)

response3=agent.invoke('Who created Langchain')
print('\nLLM answer: ',response3)

C:\Users\DELL\AppData\Local\Temp\ipykernel_9128\337605753.py:7: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent=initialize_agent(




> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? No
AI: LangChain was created by Harrison Chase.

> Finished chain.

LLM answer:  {'input': 'Who created Langchain', 'chat_history': [HumanMessage(content='Who created Langchain', additional_kwargs={}, response_metadata={}), AIMessage(content='LangChain was created by Harrison Chase.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Who created Langchain', additional_kwargs={}, response_metadata={}), AIMessage(content='LangChain was created by Harrison Chase.', additional_kwargs={}, response_metadata={})], 'output': 'LangChain was created by Harrison Chase.'}


In [9]:
compressed_docs=compressor.compress_documents(documents, query="Summarize the key points")

print("Compressed Summary: ")
for doc in compressed_docs:
    print("-", doc.page_content)

qa_prompt=PromptTemplate.from_template("Given the context below, answer the question:\n\nContext:\n{context}\n\nQuestion: {question}")

qa_chain=LLMChain(llm=llm, prompt=qa_prompt)

response=qa_chain.invoke({
    "context": "\n".join([doc.page_content for doc in compressed_docs]),
    "question": "What is the main idea of the document?"
})

print("\nAnswer to your question: ",response["text"])

Compressed Summary: 
- LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow


C:\Users\DELL\AppData\Local\Temp\ipykernel_9128\3409861475.py:9: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain=LLMChain(llm=llm, prompt=qa_prompt)



Answer to your question:  The main idea of the document is that **LangChain is a framework for building applications with Large Language Models (LLMs), highlighting its creator, key features, and common uses.**
